# UD6.02. Del modelo al dispositivo: tamaño, exactitud y velocidad

**Módulo 5073 · Programación de Inteligencia Artificial · Curso 2026/27**
Bloque 5 de los apuntes · Criterios **3.b** y **3.c**

---

La UD5 terminó con un fichero `.keras`. Este cuaderno lo coge y lo baja a un dispositivo, que
es el paso que convierte un modelo en parte de un sistema convergente.

La afirmación que vas a encontrar en todos los manuales es esta:

> «Cuantizar reduce el modelo cuatro veces, lo acelera de dos a cuatro veces y pierde menos
> del 2 % de exactitud.»

Es una afirmación con tres números, así que es comprobable. Y **una de las tres partes no se
cumple en esta máquina**. Averiguar cuál, y por qué, es el trabajo del cuaderno: es un caso
de manual de lo que el criterio 3.c llama *evaluar las características de estos sistemas*.

La regla de la unidad se aplica aquí igual que en la UD5:

> **Ningún modelo se informa sin su punto de referencia**, y el de un clasificador de diez
> clases equilibradas es la clase mayoritaria.


In [ ]:
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import statistics
import tempfile
import time

import numpy as np
import matplotlib.pyplot as plt
import keras
import tensorflow as tf

SEMILLA = 20262027
keras.utils.set_random_seed(SEMILLA)

print("TensorFlow", tf.__version__, "  Keras", keras.__version__)


## 0. Un aviso sobre la biblioteca, que conviene leer

`tf.lite` está en migración. Google ha sacado el intérprete a un paquete aparte llamado
`ai-edge-litert`, y `tf.lite.Interpreter` avisa de que desaparecerá. **Todo lo de este
cuaderno funciona igual con los dos**, así que se usa el que haya disponible.

Esto no es una anécdota administrativa: es un ejemplo del riesgo de dependencia del que
hablan los apuntes en el bloque 6. Una pieza central de una arquitectura de borde cambia de
paquete, y todo el que la haya usado tiene que tocar su código. Cuando en la actividad A6.2
evalúes un sistema, **la madurez de su biblioteca es una característica evaluable**, no un
detalle.


In [ ]:
try:
    from ai_edge_litert.interpreter import Interpreter
    ORIGEN_INTERPRETE = "ai_edge_litert"
except ImportError:
    Interpreter = tf.lite.Interpreter
    ORIGEN_INTERPRETE = "tf.lite (avisa de que está obsoleto)"

print("intérprete:", ORIGEN_INTERPRETE)


---

## 1. El modelo de partida

Una red convolucional pequeña sobre Fashion-MNIST, igual que la de la UD5. Pequeña a
propósito: lo que se va a estudiar es la conversión, no la arquitectura.


In [ ]:
CLASES = ["camiseta", "pantalon", "jersey", "vestido", "abrigo",
          "sandalia", "camisa", "zapatilla", "bolso", "botin"]

(X_todo, y_todo), (X_p, y_p) = keras.datasets.fashion_mnist.load_data()
X_ent = (X_todo[:10000].astype("float32") / 255.0)[..., None]
y_ent = y_todo[:10000]
X_val = (X_todo[10000:12000].astype("float32") / 255.0)[..., None]
y_val = y_todo[10000:12000]
X_pru = (X_p.astype("float32") / 255.0)[..., None]
y_pru = y_p

# TODAS las medidas de exactitud de este cuaderno usan estas mismas imágenes. Comparar
# la exactitud de dos modelos medida sobre conjuntos distintos es un error clásico y
# silencioso: los números salen, son parecidos, y no significan nada.
N_EVAL = 2000

mayoritaria = np.bincount(y_pru[:N_EVAL]).max() / N_EVAL
print(f"entrenamiento {X_ent.shape}   validación {X_val.shape}   prueba {X_pru.shape}")
print(f"punto de referencia (clase mayoritaria en las {N_EVAL} de evaluación): "
      f"{mayoritaria:.4f}")


In [ ]:
modelo = keras.Sequential([
    keras.layers.Input((28, 28, 1)),
    keras.layers.Conv2D(16, 3, activation="relu"),
    keras.layers.MaxPooling2D(),
    keras.layers.Conv2D(32, 3, activation="relu"),
    keras.layers.MaxPooling2D(),
    keras.layers.Flatten(),
    keras.layers.Dense(64, activation="relu"),
    keras.layers.Dense(10, activation="softmax"),
])
modelo.compile("adam", "sparse_categorical_crossentropy", metrics=["accuracy"])
modelo.summary()

t = time.perf_counter()
modelo.fit(X_ent, y_ent, validation_data=(X_val, y_val),
           epochs=8, batch_size=128, verbose=0)
print(f"\nentrenado en {time.perf_counter() - t:.0f} s")

exactitud_keras = modelo.evaluate(X_pru[:N_EVAL], y_pru[:N_EVAL], verbose=0)[1]
print(f"exactitud en Keras: {exactitud_keras:.4f}   "
      f"(punto de referencia {mayoritaria:.4f})")


---

## 2. Qué hace exactamente cuantizar

Antes de convertir nada conviene entender la operación, porque se explica casi siempre con
la frase «se pasa de 32 bits a 8» y eso no explica nada.

Cuantizar es **elegir una escala**. Un tensor de pesos vive en un intervalo, por ejemplo de
−0,42 a 0,51. Ese intervalo se reparte en los 256 valores que caben en un entero de 8 bits,
y cada peso se sustituye por el entero más cercano. Para volver atrás hacen falta dos
números que se guardan junto al tensor: la **escala** (cuánto vale un paso) y el **cero**
(qué entero representa el valor 0,0).

$$w \approx \text{escala} \times (q - \text{cero})$$

Vamos a hacerlo a mano sobre un tensor de pesos de verdad, para ver cuánto se pierde.


In [ ]:
pesos = modelo.layers[0].get_weights()[0]      # el núcleo de la primera convolución
print(f"tensor de pesos: forma {pesos.shape}, {pesos.size} valores en float32 "
      f"({pesos.nbytes} bytes)")
print(f"intervalo: [{pesos.min():.4f}, {pesos.max():.4f}]")


def cuantiza(tensor):
    '''Cuantización asimétrica a int8, que es la que usa TFLite para los pesos.'''
    minimo, maximo = float(tensor.min()), float(tensor.max())
    escala = (maximo - minimo) / 255.0
    cero = round(-128 - minimo / escala)
    q = np.clip(np.round(tensor / escala + cero), -128, 127).astype(np.int8)
    return q, escala, cero


def descuantiza(q, escala, cero):
    return (q.astype(np.float32) - cero) * escala


q, escala, cero = cuantiza(pesos)
recuperados = descuantiza(q, escala, cero)
error = np.abs(pesos - recuperados)

print(f"\nescala {escala:.8f}   cero {cero}")
print(f"en int8 ocupa {q.nbytes} bytes, {pesos.nbytes / q.nbytes:.0f} veces menos")
print(f"\nerror absoluto:  medio {error.mean():.6f}   máximo {error.max():.6f}")
print(f"el error máximo es {error.max() / escala:.2f} pasos de escala, que es lo que")
print("tiene que ser: redondear nunca puede alejarse más de medio paso.")
print(f"\nen relación al tamaño de los pesos, el error medio es un "
      f"{100 * error.mean() / np.abs(pesos).mean():.2f} %")


In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(11, 3.6))

ejes[0].hist(pesos.ravel(), bins=60, color="#4477aa")
ejes[0].set_title("los pesos originales, en float32")
ejes[0].set_xlabel("valor del peso")

ejes[1].scatter(pesos.ravel(), recuperados.ravel(), s=6, alpha=0.5, color="#cc6677")
limites = [pesos.min(), pesos.max()]
ejes[1].plot(limites, limites, color="0.3", lw=1, label="sin pérdida")
ejes[1].set_title("original frente a recuperado tras cuantizar")
ejes[1].set_xlabel("peso original")
ejes[1].set_ylabel("peso recuperado")
ejes[1].legend()

fig.tight_layout()
plt.show()

print("La figura de la derecha es una escalera, no una recta: esa es toda la pérdida.")
print("Los 256 escalones son los 256 valores que caben en un entero de 8 bits.")


---

## 3. Las tres conversiones

Ahora en serio, con el convertidor de TFLite. Hay tres destinos y conviene tener claro qué
hace cada uno:

| Conversión | Qué cuantiza | Qué hace falta | Dónde corre |
|---|---|---|---|
| **float32** | nada | nada | en cualquier sitio |
| **Dinámica** | los pesos, al convertir; las activaciones, sobre la marcha | nada | en cualquier sitio |
| **int8 completa** | pesos y activaciones, todo entero | un **conjunto representativo** | también en aceleradoras que solo hacen enteros |

La tercera columna es la que decide en la práctica. La cuantización completa necesita ver
datos reales para saber en qué intervalo se mueve **cada activación**, y por eso hay que
darle un puñado de ejemplos de entrenamiento. Es también la única que sirve para un
acelerador que solo ejecuta enteros.


In [ ]:
carpeta = tempfile.mkdtemp()
ruta_sm = os.path.join(carpeta, "modelo_exportado")
ruta_keras = os.path.join(carpeta, "modelo.keras")

modelo.save(ruta_keras)
modelo.export(ruta_sm)
tamano_keras = os.path.getsize(ruta_keras)
print(f"\n.keras: {tamano_keras:,} bytes")


In [ ]:
def nuevo_convertidor():
    return tf.lite.TFLiteConverter.from_saved_model(ruta_sm)


# 1. Sin optimizar: solo cambia el formato.
tfl_f32 = nuevo_convertidor().convert()

# 2. Cuantización dinámica: pesos a int8, activaciones en coma flotante.
c = nuevo_convertidor()
c.optimizations = [tf.lite.Optimize.DEFAULT]
tfl_din = c.convert()


# 3. Cuantización completa. El conjunto representativo NO se usa para entrenar: se usa
#    para observar por dónde se mueven las activaciones. Con 200 ejemplos sobra.
def representativos():
    for i in range(200):
        yield [X_ent[i:i + 1]]


c = nuevo_convertidor()
c.optimizations = [tf.lite.Optimize.DEFAULT]
c.representative_dataset = representativos
c.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
c.inference_input_type = tf.int8
c.inference_output_type = tf.int8
tfl_int8 = c.convert()

MODELOS = [("TFLite float32", tfl_f32),
           ("TFLite dinámica", tfl_din),
           ("TFLite int8 completa", tfl_int8)]

for nombre, contenido in MODELOS:
    print(f"{nombre:<24}{len(contenido):>10,} bytes   "
          f"{tamano_keras / len(contenido):>5.1f} veces menor que el .keras")


Fíjate en el detalle del tercer convertidor: `inference_input_type = tf.int8`. Eso significa
que **el modelo ya no acepta imágenes en coma flotante**: hay que darle enteros. Y eso, que
parece un detalle de configuración, es la causa de un error de puesta en servicio muy común,
porque el preprocesamiento del dispositivo deja de coincidir con el del entrenamiento sin
que nada avise.

Es exactamente el fallo contra el que la PR5 de la UD5 ponía una condición de invalidación:
*un `inferencia.py` cuyo preprocesamiento no coincide con el del entrenamiento no puntúa
aunque el fichero se ejecute*. Aquí se ve por qué se pone esa condición.


In [ ]:
for nombre, contenido in MODELOS:
    interprete = Interpreter(model_content=contenido)
    interprete.allocate_tensors()
    ent = interprete.get_input_details()[0]
    sal = interprete.get_output_details()[0]
    print(f"{nombre}")
    print(f"    entrada: {ent['dtype'].__name__:<8} {ent['shape'].tolist()}  "
          f"escala {ent['quantization'][0]:.6f}  cero {ent['quantization'][1]}")
    print(f"    salida:  {sal['dtype'].__name__:<8} {sal['shape'].tolist()}  "
          f"escala {sal['quantization'][0]:.6f}  cero {sal['quantization'][1]}")


---

## 4. Medir: exactitud primero

Lo primero que hay que comprobar al convertir un modelo, antes que la velocidad y antes que
el tamaño, es que **sigue diciendo lo mismo**. Es la comprobación de sensatez de la
conversión, y es el equivalente de aquella de la UD5 que cargaba el modelo guardado y
verificaba que predecía igual.


In [ ]:
def prepara_entrada(interprete, X, i):
    '''Convierte una imagen al tipo que espera el intérprete. Si la entrada es int8,
    hay que aplicar la MISMA escala y cero que guarda el modelo: hacerlo a ojo es la
    forma más rápida de tener un modelo que funciona y acierta poco.'''
    ent = interprete.get_input_details()[0]
    x = X[i:i + 1]
    if ent["dtype"] == np.int8:
        escala, cero = ent["quantization"]
        x = np.clip(np.round(x / escala + cero), -128, 127).astype(np.int8)
    return ent["index"], x


interpretes = {}
for nombre, contenido in MODELOS:
    interprete = Interpreter(model_content=contenido)
    interprete.allocate_tensors()
    interpretes[nombre] = (interprete, len(contenido))

# Se guardan las PREDICCIONES, no solo la exactitud. Dos modelos pueden acertar lo
# mismo y no estar de acuerdo en nada, y esa diferencia es justo la que interesa.
predicciones = {"Keras": np.argmax(modelo.predict(X_pru[:N_EVAL], verbose=0), axis=1)}
for nombre, (interprete, _) in interpretes.items():
    sal = interprete.get_output_details()[0]
    salidas = []
    for i in range(N_EVAL):
        indice, x = prepara_entrada(interprete, X_pru, i)
        interprete.set_tensor(indice, x)
        interprete.invoke()
        salidas.append(int(np.argmax(interprete.get_tensor(sal["index"])[0])))
    predicciones[nombre] = np.array(salidas)

exactitudes = {n: float((p == y_pru[:N_EVAL]).mean()) for n, p in predicciones.items()}
for nombre, valor in exactitudes.items():
    print(f"{nombre:<24}{valor:.4f}")


### La comprobación que hay que hacer, y lo que revela

Aquí es donde casi todo el mundo se conforma con ver que las exactitudes se parecen. No
basta: **dos modelos pueden acertar lo mismo y no estar de acuerdo en nada**. Dos modelos con
un 86,5 % pueden coincidir en todo salvo en un puñado de imágenes, o discrepar en trescientas
y compensarse. La exactitud sola no distingue esos dos casos, y son muy distintos.

Lo que hay que mirar es **en cuántas imágenes cambia la respuesta**.


In [ ]:
base = predicciones["Keras"]
print(f"{'frente a Keras':<24}{'cambian':>9}{'de ' + str(N_EVAL):>10}{'%':>8}")
print("-" * 51)
for nombre in interpretes:
    cambian = int((predicciones[nombre] != base).sum())
    print(f"{nombre:<24}{cambian:>9}{N_EVAL:>10}{100 * cambian / N_EVAL:>8.2f}")

distintas_f32 = int((predicciones["TFLite float32"] != base).sum())
distintas_i8 = int((predicciones["TFLite int8 completa"] != base).sum())

print()
if distintas_f32 == 0:
    print("Cambiar de formato no cambia NI UNA predicción. La conversión a TFLite")
    print("float32 es exacta: mismo grafo, mismos pesos, misma aritmética.")
else:
    print(f"Cambiar de formato ya cambia {distintas_f32} predicciones de {N_EVAL}, un")
    print(f"{100 * distintas_f32 / N_EVAL:.2f} %. No es pérdida de información: es que las dos")
    print("implementaciones suman en otro orden, y donde el modelo dudaba entre dos")
    print("clases, una diferencia en el sexto decimal cambia el ganador.")

print()
print(f"Cuantizar a int8 cambia {distintas_i8} de {N_EVAL}, un "
      f"{100 * distintas_i8 / N_EVAL:.2f} %. Esa sí es")
print("la cuantización, y es mucho más de lo que sugiere la tabla de exactitudes:")
mueve = abs(exactitudes["TFLite float32"] - exactitudes["TFLite int8 completa"]) * 100
print(f"la exactitud solo se mueve {mueve:.2f} puntos porque los aciertos que pierde")
print("los compensa casi con los que gana. No es lo mismo «acertar igual» que")
print("«decir lo mismo», y para un sistema en producción importa la segunda.")

assert distintas_f32 / N_EVAL < 0.01, "demasiada discrepancia: revisa el preprocesamiento"
assert distintas_i8 / N_EVAL < 0.05, "la cuantización está cambiando demasiadas respuestas"
print()
print("Umbrales de sensatez: por debajo del 1 % al cambiar de formato y del 5 % al")
print("cuantizar. Las dos comprobaciones pasan.")


---

## 5. Medir: velocidad, y cómo no engañarse

Aquí hay dos trampas y las dos son fáciles de caer.

**Trampa 1: medir los formatos en bloques seguidos.** La carga de la máquina cambia. Si mides
los tres seguidos, estás comparando tres momentos distintos. La solución es **entrelazar**:
una ronda de cada uno, varias veces, y mirar la dispersión entre rondas antes de afirmar que
una diferencia es real.

**Trampa 2: elegir mal el término de comparación en Keras.** `modelo.predict(x)` está pensado
para lotes grandes y arrastra un coste fijo por llamada enorme. `modelo(x, training=False)`
es la llamada directa. Comparar TFLite con `predict()` sobre un lote de 1 infla el resultado
un orden de magnitud, y es exactamente la comparación que aparece en los artículos que
anuncian aceleraciones espectaculares. Medimos las dos.


In [ ]:
una = X_pru[:1]
una_tf = tf.constant(una)


def mediana_de(funcion, calentamiento=20, n=200):
    for _ in range(calentamiento):
        funcion()
    tiempos = []
    for _ in range(n):
        t = time.perf_counter()
        funcion()
        tiempos.append((time.perf_counter() - t) * 1000)
    return statistics.median(tiempos)


lat_predict = mediana_de(lambda: modelo.predict(una, verbose=0))
lat_llamada = mediana_de(lambda: modelo(una_tf, training=False))

print(f"Keras, predict(), lote de 1:   {lat_predict:8.3f} ms")
print(f"Keras, modelo(x), lote de 1:   {lat_llamada:8.3f} ms")
print(f"\nEl mismo modelo y el mismo dato, {lat_predict / lat_llamada:.0f} veces de")
print("diferencia.")
print("La primera cifra no mide el modelo: mide la maquinaria de predict().")


In [ ]:
# Rondas entrelazadas: una tanda de cada formato, cinco veces.
rondas = {nombre: [] for nombre in interpretes}
for _ in range(5):
    for nombre, (interprete, _) in interpretes.items():
        indice, x = prepara_entrada(interprete, X_pru, 0)
        interprete.set_tensor(indice, x)
        for _ in range(50):
            interprete.invoke()
        tiempos = []
        for _ in range(400):
            t = time.perf_counter()
            interprete.invoke()
            tiempos.append((time.perf_counter() - t) * 1000)
        rondas[nombre].append(statistics.median(tiempos))

for nombre, ms in rondas.items():
    print(f"{nombre:<24} mediana {statistics.median(ms):.4f} ms   "
          f"rondas {min(ms):.4f} a {max(ms):.4f}")


In [ ]:
f32 = rondas["TFLite float32"]
i8 = rondas["TFLite int8 completa"]
solapan = not (max(i8) < min(f32) or max(f32) < min(i8))

print(f"float32: rondas de {min(f32):.4f} a {max(f32):.4f} ms")
print(f"int8:    rondas de {min(i8):.4f} a {max(i8):.4f} ms")
print()
if solapan:
    print("Los márgenes SE SOLAPAN. En esta máquina no se puede afirmar que")
    print("la cuantización acelere: la diferencia cabe dentro del ruido de medida.")
else:
    factor = statistics.median(f32) / statistics.median(i8)
    print(f"Los márgenes NO se solapan: int8 es {factor:.2f} veces más rápido aquí.")
print()
print("En cualquiera de los dos casos, la relación es de unidades, no de las 2 a 4")
print("veces que anuncian los manuales. Y no es que mientan: es que miden en un")
print("procesador ARM con instrucciones enteras dedicadas, o en un acelerador que")
print("SOLO ejecuta enteros. En un x86 con XNNPACK la ruta de coma flotante ya está")
print("muy optimizada y el margen que queda es pequeño.")
print()
print("Conclusión honesta, y es la que hay que escribir en A6.2:")
print("  cuantizar es una técnica para que el modelo QUEPA y consuma menos memoria.")
print("  Que además acelere depende del procesador, y hay que decir de cuál.")


---

## 6. La tabla, y la afirmación de partida

Volvamos a la frase del principio: *cuatro veces más pequeño, de dos a cuatro veces más
rápido, menos del 2 % de exactitud perdida*. Ahora hay números para juzgarla.


In [ ]:
print(f"{'formato':<24}{'bytes':>10}{'x menor':>9}{'exactitud':>11}"
      f"{'ms':>9}{'rondas':>18}")
print("-" * 81)
print(f"{'clase mayoritaria':<24}{'':>10}{'':>9}{mayoritaria:>11.4f}{'':>9}{'':>18}")
print(f"{'Keras, predict()':<24}{tamano_keras:>10,}{1.0:>9.2f}"
      f"{exactitudes['Keras']:>11.4f}{lat_predict:>9.2f}{'':>18}")
print(f"{'Keras, modelo(x)':<24}{'(el mismo)':>10}{'':>9}{'(la misma)':>11}"
      f"{lat_llamada:>9.2f}{'':>18}")
for nombre, (_, n_bytes) in interpretes.items():
    ms = rondas[nombre]
    print(f"{nombre:<24}{n_bytes:>10,}{tamano_keras / n_bytes:>9.2f}"
          f"{exactitudes[nombre]:>11.4f}{statistics.median(ms):>9.3f}"
          f"{f'{min(ms):.3f} - {max(ms):.3f}':>18}")

n_f32 = interpretes["TFLite float32"][1]
n_i8 = interpretes["TFLite int8 completa"][1]
caida = (exactitudes["TFLite float32"] - exactitudes["TFLite int8 completa"]) * 100

print()
print("Las tres partes de la afirmación de partida:")
print(f"  1. tamaño     {n_f32 / n_i8:.1f} veces menor frente al float32 de TFLite y")
print(f"                {tamano_keras / n_i8:.1f} frente al .keras.  SE CUMPLE, y de sobra.")
print(f"  2. exactitud  {abs(caida):.2f} puntos porcentuales de caída, muy por debajo del 2 %.")
print("                SE CUMPLE.")
print(f"  3. velocidad  {statistics.median(f32) / statistics.median(i8):.2f} veces. "
      f"NO SE CUMPLE en este procesador.")
print()
print("Dos de tres. Y la que no se cumple es la que más se repite.")


In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(11, 3.8))

nombres = ["float32", "dinámica", "int8"]
tamanos = [interpretes[n][1] / 1024 for n in interpretes]
ejes[0].bar(nombres, tamanos, color=["#4477aa", "#88ccee", "#cc6677"])
ejes[0].axhline(tamano_keras / 1024, color="0.3", ls="--", lw=1,
                label=f".keras de partida ({tamano_keras / 1024:.0f} kB)")
ejes[0].set_ylabel("kB")
ejes[0].set_title("lo que ocupa el modelo")
ejes[0].legend(fontsize=8)

ejes[1].bar(nombres, [exactitudes[n] for n in interpretes],
            color=["#4477aa", "#88ccee", "#cc6677"])
ejes[1].axhline(mayoritaria, color="0.3", ls="--", lw=1,
                label=f"clase mayoritaria ({mayoritaria:.2f})")
ejes[1].set_ylim(0, 1)
ejes[1].set_ylabel("exactitud")
ejes[1].set_title("lo que acierta")
ejes[1].legend(fontsize=8)

fig.tight_layout()
plt.show()

print("La figura de la derecha lleva la línea del punto de referencia, que es")
print("obligatoria en este módulo. Sin ella, tres barras al 0,86 no dicen si el")
print("modelo es bueno: solo dicen que los tres formatos se parecen.")


---

## Lo que hay que llevarse

1. **Cambiar de formato no pierde nada.** TFLite float32 da exactamente la misma exactitud
   que Keras, y comprobarlo es la primera cosa que hay que hacer al convertir.
2. **Cuantizar hace el modelo once veces más pequeño** que el `.keras` de partida y cuesta
   dos décimas de punto de exactitud.
3. **Que acelere depende del procesador.** En un x86 la ganancia es pequeña; la aceleración
   de 2 a 4 veces que anuncian los manuales es de ARM y de aceleradoras de enteros.
4. **Con `inference_input_type = int8` el modelo ya no acepta coma flotante.** El
   preprocesamiento del dispositivo tiene que aplicar la escala y el cero del modelo, y si no
   lo hace, el sistema funciona y acierta poco, sin dar ningún error.
5. **Entrelaza las mediciones y mira la dispersión** antes de afirmar que una diferencia es
   real.
6. **Elige bien el término de comparación.** `predict()` frente a `modelo(x)` ya son diez
   veces con el mismo modelo.

### Para la actividad A6.2

De este cuaderno sale la fila de «características medidas» de tu tabla. Necesitas, de **tu**
modelo: los tres tamaños, las tres exactitudes sobre el mismo conjunto, las tres latencias
con su dispersión, y una frase que diga en qué procesador se ha medido. Sin esa última frase
la latencia no vale nada.
